# 05 -- Fetch Staff Subscriptions

Staff-migration variant of `05_Fetch_Subscriptions.ipynb`. Loads the exact
active-subscription snapshot `04_Create_Staff_Accounts.ipynb` already pulled
(`staff_subscriptions_active_raw` -- not re-queried here, so this can't drift
from the set that decided which accounts got created), then for each one:

1. Resolves `TargetAccountNumber` -- a straight join of the subscription's
   `AccountCode` against `04`'s `GeneratedAccountNumber` results
   (`load_staff_account_number_map()`). No bucket-account routing, no
   `Reference`-based special cases -- every staff subscription goes onto its
   own real account, or nowhere at all if that account isn't ready.
2. Looks up its real address + radius username from Voyager
   (`get_voyager_address`), exactly as `05_Fetch_Subscriptions.ipynb` does.

Output feeds `06_Create_Staff_Addresses.ipynb` and
`07_Create_Staff_Subscription_Orders.ipynb`.

## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("fetch_staff_subscriptions")

TEST_ROW_LIMIT = 5  # Testing limiter -- set to None for a full run.


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 27
python-dotenv could not parse statement starting at line 32
python-dotenv could not parse statement starting at line 38
python-dotenv could not parse statement starting at line 44


In [2]:
# HARDCODED_TOKEN = "c5bf2481-f3fd-498b-a260-27c6e783782f"

# token_manager._token = HARDCODED_TOKEN
# token_manager._expires_at = datetime.now() + timedelta(hours=1)  # adjust to match the real token's actual TTL

## 2. Load the active-subscriptions snapshot

Same rows `04_Create_Staff_Accounts.ipynb` used to decide which accounts to
create -- loaded from disk, not re-queried, so the two notebooks can't see
different data if the underlying MySQL table changes in between runs.

In [3]:
df_subscriptions = load_df("staff_subscriptions_active_raw")
logger.info(f"Loaded {len(df_subscriptions):,} active staff subscriptions from 04's snapshot")

if TEST_ROW_LIMIT is not None:
    df_subscriptions = df_subscriptions.head(TEST_ROW_LIMIT)  # Testing limiter -- remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active -- trimmed to {len(df_subscriptions):,} rows")

df_subscriptions.head()


2026-07-31 11:29:16,816 [INFO] Loaded 230 active staff subscriptions from 04's snapshot
2026-07-31 11:29:16,818 [INFO] TEST_ROW_LIMIT active -- trimmed to 5 rows


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,CircuitType,Server,CustomerSuppliedReference,_notforreports_VoyagerOrderHistory,_notforreports_LegacyServiceDescription,NextPlanCode,NextPlanStartDate,NextQuantity,NextCustomPrice,SalesAgentCode
0,449599,2023-08-15 10:22:05,167280,vBill,94080157,IP Voice,V112301122,b09c470a-bc44-06a9-e778-92d913d54f34,2014-05-28,NaN,...,NaN,NaN,NaN,2023-08-12: Change to VV Premium [CAS-719102-H...,NaN,NaN,NaN,NaN,NaN,NaN
1,449624,2021-09-03 01:21:52,167285,vBill,94080157,Email,V112301171,mannan@actrix.co.nz,2014-05-28,NaN,...,NaN,NaN,NaN,NaN,Actrix: Email,NaN,NaN,NaN,NaN,NaN
2,449629,2024-08-13 12:23:17,167286,vBill,94080157,Broadband - Fibre,V112301189,alfaris.ali@vygr.net,2014-05-28,NaN,...,NaN,NaN,NaN,NaN,Actrix: Residential Unlimited UFB 30/10; Migra...,NaN,NaN,NaN,NaN,NaN
3,1257700957,2022-01-03 01:12:11,220983,vBill,94080157,Hardware Rental,V112727565,J3N7S18518922708,2021-12-05,NaN,...,NaN,NaN,NaN,PROV-164305,NaN,NaN,NaN,NaN,NaN,NaN
4,4112118871,2025-10-08 01:25:42,259677,vBill,94080157,Broadband - Fibre,V113052864,alfaris.ali17@vygr.net,2025-10-06,NaN,...,UFB 100/20,NaN,NaN,PROV-215979,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Resolve target account for every subscription

Direct join on `AccountCode` -- `load_staff_account_number_map()` returns
`{AccountCode: GeneratedAccountNumber}` for every staff account that both
(a) was actually created (`status == "created"`, not `"exists"` -- see the
idempotency note in `04_Create_Staff_Accounts.ipynb`) and (b) had its
OneBill-assigned account number successfully extracted from the create
response.

Any subscription whose account isn't in that map (creation failed, or
`GeneratedAccountNumber` couldn't be extracted) gets `TargetAccountNumber =
NaN` here -- there's no bucket-account fallback for staff, so those rows are
flagged and excluded rather than silently rerouted. Check the "unresolved"
table below before continuing to `06_Create_Staff_Addresses.ipynb`.

In [4]:
own_account_map = load_staff_account_number_map()  # {AccountCode: GeneratedAccountNumber}
logger.info(f"{len(own_account_map):,} staff accounts have a confirmed OneBill accountNumber from 04's results")

df_subscriptions["AccountCode"] = df_subscriptions["AccountCode"].astype(str)
df_subscriptions["TargetAccountKey"] = "own_account"
df_subscriptions["TargetAccountNumber"] = df_subscriptions["AccountCode"].map(own_account_map)

unresolved = df_subscriptions[df_subscriptions["TargetAccountNumber"].isna()]
if not unresolved.empty:
    logger.warning(
        f"{len(unresolved):,} / {len(df_subscriptions):,} subscriptions have no resolved TargetAccountNumber "
        f"-- their account either failed to create or its GeneratedAccountNumber couldn't be extracted "
        f"(see 04_Create_Staff_Accounts.ipynb's failure / missing-number summaries). Dropping them here; "
        f"re-run 04 for these AccountCodes and come back to this notebook once they resolve."
    )

df_subscriptions = df_subscriptions[df_subscriptions["TargetAccountNumber"].notna()].copy()
logger.info(f"{len(df_subscriptions):,} subscriptions have a resolved TargetAccountNumber and will proceed")

unresolved[["SubscriptionUSN", "AccountCode"]].head(20) if not unresolved.empty else unresolved


2026-07-31 11:29:16,873 [INFO] 4 staff accounts have a confirmed OneBill accountNumber from 04's results
2026-07-31 11:29:16,879 [WARNING] 5 / 5 subscriptions have no resolved TargetAccountNumber -- their account either failed to create or its GeneratedAccountNumber couldn't be extracted (see 04_Create_Staff_Accounts.ipynb's failure / missing-number summaries). Dropping them here; re-run 04 for these AccountCodes and come back to this notebook once they resolve.
2026-07-31 11:29:16,883 [INFO] 0 subscriptions have a resolved TargetAccountNumber and will proceed


,SubscriptionUSN,AccountCode
0,V112301122,94080157
1,V112301171,94080157
2,V112301189,94080157
3,V112727565,94080157
4,V113052864,94080157


## 4. Voyager address lookup (circuits + address-search)

Identical to `05_Fetch_Subscriptions.ipynb` step 4 -- `SupplierServiceID` ->
`GET .../fibre/v1/circuits/{id}` (gives `radiusUsers[0]` + `locationId`) ->
`GET .../address-search/v3/addresses/id/{locationId}` (gives the actual
street address, city, postcode, region). Region name mapped to a 3-char ISO
code via `NZ_Regions.xlsx`.

In [5]:
if VOYAGER_CCP_KEY is None or VOYAGER_PARTNER_ID is None:
    logger.warning("VOYAGER_CCP_KEY / VOYAGER_PARTNER_ID not set -- every Voyager lookup below will fail. Set them in .env.")

blank_supplier_ids = df_subscriptions["SupplierServiceID"].isna() | (df_subscriptions["SupplierServiceID"].astype(str).str.strip() == "")
if blank_supplier_ids.all():
    logger.warning(
        "SupplierServiceID is blank for EVERY subscription in this batch -- the Voyager circuits lookup "
        "can't run at all without it, so every ParsedAddress_* field below will be None. Check the MySQL "
        "source data / STAFF_ACTIVE_SUBSCRIPTIONS_QUERY in 04_Create_Staff_Accounts.ipynb before re-running."
    )
elif blank_supplier_ids.any():
    logger.warning(f"{blank_supplier_ids.sum():,} / {len(df_subscriptions):,} subscriptions have a blank SupplierServiceID")

voyager_session = new_voyager_session(max_workers=MAX_WORKERS)


def _lookup_row(supplier_service_id):
    return get_voyager_address(voyager_session, supplier_service_id)


logger.info(f"Looking up Voyager address for {len(df_subscriptions):,} subscriptions with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    voyager_results = list(executor.map(_lookup_row, df_subscriptions["SupplierServiceID"]))

existing_parsed_cols = [c for c in df_subscriptions.columns if c.startswith("ParsedAddress_")]
if existing_parsed_cols:
    df_subscriptions = df_subscriptions.drop(columns=existing_parsed_cols)  # safe to re-run this cell

address_parts = pd.DataFrame(voyager_results, index=df_subscriptions.index)
address_parts = address_parts.add_prefix("ParsedAddress_")
df_subscriptions = pd.concat([df_subscriptions, address_parts], axis=1)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    logger.warning(f"{len(unparsed):,} subscriptions could not be resolved to a Voyager address")
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info("Error breakdown:\n" + error_summary.to_string(index=False))

df_subscriptions[[
    "SubscriptionLabel", "SupplierServiceID",
    "ParsedAddress_addLine1", "ParsedAddress_addLine2", "ParsedAddress_city",
    "ParsedAddress_postcode", "ParsedAddress_region_iso", "ParsedAddress_region_code_raw", "ParsedAddress_radius_user",
    "ParsedAddress_parsed_ok", "ParsedAddress_error",
]].head(20)


2026-07-31 11:29:26,434 [WARNING] SupplierServiceID is blank for EVERY subscription in this batch -- the Voyager circuits lookup can't run at all without it, so every ParsedAddress_* field below will be None. Check the MySQL source data / STAFF_ACTIVE_SUBSCRIPTIONS_QUERY in 04_Create_Staff_Accounts.ipynb before re-running.
2026-07-31 11:29:26,436 [INFO] Looking up Voyager address for 0 subscriptions with 3 workers...


KeyError: 'ParsedAddress_parsed_ok'

## 5. Save

In [ ]:
save_df("staff_subscriptions_resolved", df_subscriptions)
